# Predicting Irrigation Need

Dataset: https://www.kaggle.com/competitions/playground-series-s6e4

Can we predict irrigation based on soil, weather and crop indicators?

Notes:
ensemble method does well at averaging the noise of the models.

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px


def kaggle_dataset(url):
    dataset_path = kagglehub.competition_download(url)
    train_df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
    test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    return train_df, test_df


def submit_to_kaggle(model, X, y, test_data, file_name, id_col, target_col) -> None:
    kaggle_model = model.fit(X, y)
    kaggle_predictions = kaggle_model.predict(test_data)
    kaggle_submission_df = pd.DataFrame({
        id_col: test_data[id_col],
        target_col: kaggle_predictions
    })
    kaggle_submission_df.to_csv(file_name, index=False, header=True)

# !kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission_3.csv -m "handling categorical data and pca"
# !kaggle competitions submissions -c house-prices-advanced-regression-techniques

In [ ]:
train, test = kaggle_dataset('playground-series-s6e4')
train

In [ ]:
train
train.Irrigation_Need.value_counts()

## Let's Do Some EDA

1. The Data [x]
    - Missing Values
    - Data Types
    - Errors
2. Handling Missing Data [x]
    - Why is the data missing? Missingness.
    - Remove or Impute?
    - Suitable Imputation Method
    - Make a note of any bias the missing data may introduce.
3. Data Distribution
    - Measure of central tendancy
    - Measure virability and standard deviation
    - Skewness and kurtosis
    - Outliers and Anomalies
4. Data Transformations
    - Scaling or Normalization
    - Encoding categorical data (one-hot or label encoding)
    - Mathematical transformations
    - Creating new features
    - Aggregating data
5. Data Relationships
    - Categorical variables

In [ ]:
import matplotlib.pyplot as plt
import math
import numpy as np

plt.style.use('_mpl-gallery')

# Automatically find all numeric columns in the dataset
numeric_cols = train.select_dtypes(include=['number']).columns.tolist()
num_plots = len(numeric_cols)

# 3. Calculate dynamic grid size (let's fix it to 2 columns, and calculate needed rows)
cols = 2
rows = math.ceil(num_plots / cols)

# Create the figure and axes
fig, axs = plt.subplots(nrows=rows, ncols=cols, figsize=(10, 4 * rows))

# Flatten the axes array (turns a 2x2 grid into a simple list of 4 items).
# This makes it MUCH easier to loop through!
# Note: If num_plots is 1, axs isn't an array, so we wrap it to be safe.
if num_plots > 1:
    axs = axs.flatten()
else:
    axs = [axs]

category_order = ['Low', 'Medium', 'High']

# 4. Loop through the numeric columns and plot
for i, col in enumerate(numeric_cols):
    ax = axs[i]

    # Pure matplotlib needs a list of data arrays, one for each box
    # This list comprehension filters the data by 'low', then 'medium', then 'high'
    plot_data = [train[train["Irrigation_Need"] == cat][col] for cat in category_order]

    # Create the plot
    ax.boxplot(plot_data, tick_labels=category_order)

    # Formatting
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel('Category')
    ax.set_ylabel(col)

# 5. Clean up any leftover empty subplots
# If you have 3 plots but a 2x2 grid, the 4th slot will be blank. This hides it.
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()

In [ ]:
# sklearn and specific project imports, update this cell with new import as required
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
train.select_dtypes(str).nunique()
# float64 and str. The id column is in64 but that should be removed/contain no predictive information.


## Univariate Analysis: Numeric Distributions


In [ ]:
numeric_cols = train.select_dtypes(include="number").columns.tolist()
numeric_cols

## Handling Categorical Data
On first inspection the target variable consists of three values, Low, Medium and High. This feels like ordinal data and could be transformed in the values 1, 2 and 3.


In [ ]:
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder())
])

## Handling Numeric Data

Create a pipeline to handle numeric data. In this case we'll be scaling.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [ ]:
# Set benchmark to some linear model etc. What features could be more powerful to the model?

In [ ]:
categorical_cols = train.select_dtypes(include=[str]).columns.drop('Irrigation_Need')
print(categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, train.select_dtypes(include=[float]).columns),
        ('cat', categorical_pipeline, categorical_cols),
    ])

In [ ]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier())
    ])

In [ ]:
y = train.Irrigation_Need
X = train.drop('Irrigation_Need', axis=1)

In [ ]:
results = cross_validate(pipeline, X, y, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1, return_estimator=True)

In [ ]:
# Get feature names from the first fold's fitted preprocessor
fitted_pipeline = results['estimator'][0]
feature_names = fitted_pipeline['preprocessor'].get_feature_names_out()

# Average importances across all CV folds
importances = np.mean(
    [est['classifier'].feature_importances_ for est in results['estimator']],
    axis=0
)

feature_importance_df = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.subplots(1, 1, figsize=(10, 6))
plt.barh(
    feature_importance_df.feature,
    feature_importance_df.importance,
    color='royalblue'
)

## Baseline model created
Let's see if we can improve on this.

In [ ]:
train

In [ ]:
# TODO: NOTE TO SELF TRY THE DIFFERENT ENCODING OF IRRIGATION_NEED AND SEE WHAT HAPPENS!!!! TRY TO RUN DIFFERENT EXPERIMENTS

In [ ]:
preprocessing_pipe = pipeline[:-1]  # Everything except the classifier

In [ ]:
preprocessing_pipe.fit_transform(X, y)

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px


X_transformed = preprocessing_pipe.fit_transform(X)

pca = PCA(n_components=3)
components = pca.fit_transform(X_transformed)

total_var = pca.explained_variance_ratio_.sum() * 100

fig = px.scatter_3d(
    components,
    x=0, y=1, z=2,
    color=y.values,
    title=f'Total explained variance: {total_var:.2f}%',
    labels={'0': 'PC1', '1': 'PC2', '2': 'PC3', 'color': 'Irrigation_Need'}
)
fig.show()

In [ ]:
submit_to_kaggle(pipeline, X, y, test, 'submission.csv', 'id', 'Irrigation_Need')

In [ ]:
# !kaggle competitions submit -c playground-series-s6e4 -f submission.csv -m "Initial decision tree submission"

In [ ]:
!kaggle competitions submissions -c playground-series-s6e4